# 5주차 · 준비된 테이블 5개 직접 적재

파일 → Drive에 사본 저장 후 위에서 아래로 실행합니다. 공용 서버 비밀번호 없이 개인 SQLite DB를 만듭니다. 원본 SQL 파일은 customers → products → orders → order_items → logins 순서로 실행합니다.

In [ ]:
import sqlite3
import requests
import pandas as pd

# 개인 메모리 DB: 이 셀을 다시 실행하면 새 DB로 시작합니다.
con = sqlite3.connect(":memory:")
con.executescript('PRAGMA foreign_keys = ON;\n\nCREATE TABLE customers (\n    customer_id INTEGER PRIMARY KEY,\n    name TEXT, region TEXT, joined_date TEXT\n);\nCREATE TABLE products (\n    product_id INTEGER PRIMARY KEY,\n    name TEXT, category TEXT, price INTEGER\n);\nCREATE TABLE orders (\n    order_id INTEGER PRIMARY KEY,\n    customer_id INTEGER REFERENCES customers(customer_id),\n    order_date TEXT, order_total INTEGER\n);\nCREATE TABLE order_items (\n    item_id INTEGER PRIMARY KEY,\n    order_id INTEGER REFERENCES orders(order_id),\n    product_id INTEGER REFERENCES products(product_id),\n    qty INTEGER, unit_price INTEGER\n);\nCREATE TABLE logins (\n    login_id INTEGER PRIMARY KEY,\n    customer_id INTEGER REFERENCES customers(customer_id),\n    login_date TEXT\n);')
base = "https://raw.githubusercontent.com/team-datapopcorn/bdai13-data-handling-weekly/master/"
tables = ["customers", "products", "orders", "order_items", "logins"]

try:
    for table in tables:
        response = requests.get(base + f"insert_{table}.sql", timeout=30)
        response.raise_for_status()
        sql = response.text.replace(f"bdai13.{table}", table)
        con.execute(sql)
    con.commit()
except Exception:
    con.rollback()
    raise

counts = pd.DataFrame([
    {"table_name": t, "row_count": con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]}
    for t in tables
])
display(counts)
assert con.execute("PRAGMA foreign_key_check").fetchall() == []
print("외래 키 연결 검증 완료")

## 기대 행 수

- customers: 400행
- products: 10행
- orders: 713행
- order_items: 1,782행
- logins: 1,953행

저장소 원본 파일 기준입니다. foreign_key_check 결과는 0행이어야 합니다.

In [ ]:
pd.read_sql_query("SELECT * FROM orders ORDER BY order_id LIMIT 5", con)

## JOIN과 집계
각 주문에 고객이 1명 연결되는지, 주문 건수와 매출 합이 그대로인지 확인합니다.

In [ ]:
pd.read_sql_query("""
SELECT c.region, COUNT(*) AS order_count, SUM(o.order_total) AS total_sales
FROM orders o JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.region ORDER BY c.region
""", con)

## 선택 · DB 파일 다운로드
개인 DB를 내려받아 DBeaver의 SQLite 연결에서 열 수 있습니다.

In [ ]:
from google.colab import files
with sqlite3.connect("caramelpopcorn_practice.db") as saved:
    con.backup(saved)
files.download("caramelpopcorn_practice.db")